## Step 1 — Native TRL + environment_factory + QLoRA

### 1. インストール

In [ ]:
%%bash
pip uninstall -y unsloth unsloth_zoo

pip install -U \
  "transformers>=5.2.0" \
  git+https://github.com/huggingface/trl.git@main \
  accelerate \
  datasets \
  peft \
  bitsandbytes \
  jmespath \
  langchain \
  langchain-core \
  langchain-mcp-adapters \
  packaging \
  rich

In [ ]:
# from dotenv import load_dotenv
# load_dotenv(".env", override=True)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = "<HF_TOKEN>"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

### 2. データセット

In [ ]:
%%writefile deepwiki_grpo_dataset_300.jsonl
{"id": "deepwiki-0001", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "architecture", "must_mention": ["RL-Factory", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0002", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "execution_flow", "must_mention": ["RL-Factory", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0003", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "dependencies", "must_mention": ["RL-Factory", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0004", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "configuration", "must_mention": ["RL-Factory", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0005", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "testing", "must_mention": ["RL-Factory", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0006", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "training_or_core_algorithm", "must_mention": ["RL-Factory", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0007", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "extension_points", "must_mention": ["RL-Factory", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0008", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "debugging", "must_mention": ["RL-Factory", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0009", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "api_comparison", "must_mention": ["RL-Factory", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0010", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Simple-Efficient/RL-Factory リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "Simple-Efficient/RL-Factory", "domain": "RL・エージェント学習", "task_type": "onboarding", "must_mention": ["RL-Factory", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0011", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "architecture", "must_mention": ["trl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0012", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "execution_flow", "must_mention": ["trl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0013", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "dependencies", "must_mention": ["trl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0014", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "configuration", "must_mention": ["trl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0015", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "testing", "must_mention": ["trl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0016", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "training_or_core_algorithm", "must_mention": ["trl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0017", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "extension_points", "must_mention": ["trl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0018", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "debugging", "must_mention": ["trl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0019", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "api_comparison", "must_mention": ["trl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0020", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/trl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/trl", "domain": "Transformer強化学習", "task_type": "onboarding", "must_mention": ["trl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0021", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "architecture", "must_mention": ["transformers", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0022", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "execution_flow", "must_mention": ["transformers", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0023", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "dependencies", "must_mention": ["transformers", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0024", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "configuration", "must_mention": ["transformers", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0025", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "testing", "must_mention": ["transformers", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0026", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "training_or_core_algorithm", "must_mention": ["transformers", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0027", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "extension_points", "must_mention": ["transformers", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0028", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "debugging", "must_mention": ["transformers", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0029", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "api_comparison", "must_mention": ["transformers", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0030", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/transformers リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/transformers", "domain": "基盤モデル・推論", "task_type": "onboarding", "must_mention": ["transformers", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0031", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "architecture", "must_mention": ["peft", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0032", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "execution_flow", "must_mention": ["peft", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0033", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "dependencies", "must_mention": ["peft", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0034", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "configuration", "must_mention": ["peft", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0035", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "testing", "must_mention": ["peft", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0036", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "training_or_core_algorithm", "must_mention": ["peft", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0037", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "extension_points", "must_mention": ["peft", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0038", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "debugging", "must_mention": ["peft", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0039", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "api_comparison", "must_mention": ["peft", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0040", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/peft リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/peft", "domain": "Parameter-Efficient Fine-Tuning", "task_type": "onboarding", "must_mention": ["peft", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0041", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "architecture", "must_mention": ["accelerate", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0042", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "execution_flow", "must_mention": ["accelerate", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0043", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "dependencies", "must_mention": ["accelerate", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0044", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "configuration", "must_mention": ["accelerate", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0045", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "testing", "must_mention": ["accelerate", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0046", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "training_or_core_algorithm", "must_mention": ["accelerate", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0047", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "extension_points", "must_mention": ["accelerate", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0048", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "debugging", "must_mention": ["accelerate", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0049", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "api_comparison", "must_mention": ["accelerate", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0050", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/accelerate リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/accelerate", "domain": "分散学習", "task_type": "onboarding", "must_mention": ["accelerate", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0051", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "architecture", "must_mention": ["datasets", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0052", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "execution_flow", "must_mention": ["datasets", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0053", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "dependencies", "must_mention": ["datasets", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0054", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "configuration", "must_mention": ["datasets", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0055", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "testing", "must_mention": ["datasets", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0056", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "training_or_core_algorithm", "must_mention": ["datasets", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0057", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "extension_points", "must_mention": ["datasets", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0058", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "debugging", "must_mention": ["datasets", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0059", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "api_comparison", "must_mention": ["datasets", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0060", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "huggingface/datasets リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "huggingface/datasets", "domain": "データセット処理", "task_type": "onboarding", "must_mention": ["datasets", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0061", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "architecture", "must_mention": ["unsloth", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0062", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "execution_flow", "must_mention": ["unsloth", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0063", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "dependencies", "must_mention": ["unsloth", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0064", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "configuration", "must_mention": ["unsloth", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0065", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "testing", "must_mention": ["unsloth", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0066", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "training_or_core_algorithm", "must_mention": ["unsloth", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0067", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "extension_points", "must_mention": ["unsloth", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0068", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "debugging", "must_mention": ["unsloth", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0069", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "api_comparison", "must_mention": ["unsloth", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0070", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "unslothai/unsloth リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "unslothai/unsloth", "domain": "高速ファインチューニング", "task_type": "onboarding", "must_mention": ["unsloth", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0071", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "architecture", "must_mention": ["langchain", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0072", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "execution_flow", "must_mention": ["langchain", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0073", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "dependencies", "must_mention": ["langchain", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0074", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "configuration", "must_mention": ["langchain", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0075", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "testing", "must_mention": ["langchain", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0076", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "training_or_core_algorithm", "must_mention": ["langchain", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0077", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "extension_points", "must_mention": ["langchain", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0078", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "debugging", "must_mention": ["langchain", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0079", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "api_comparison", "must_mention": ["langchain", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0080", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langchain リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/langchain", "domain": "LLMアプリケーション", "task_type": "onboarding", "must_mention": ["langchain", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0081", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "architecture", "must_mention": ["langgraph", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0082", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "execution_flow", "must_mention": ["langgraph", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0083", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "dependencies", "must_mention": ["langgraph", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0084", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "configuration", "must_mention": ["langgraph", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0085", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "testing", "must_mention": ["langgraph", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0086", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "training_or_core_algorithm", "must_mention": ["langgraph", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0087", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "extension_points", "must_mention": ["langgraph", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0088", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "debugging", "must_mention": ["langgraph", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0089", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "api_comparison", "must_mention": ["langgraph", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0090", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/langgraph リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/langgraph", "domain": "ステートフルエージェント", "task_type": "onboarding", "must_mention": ["langgraph", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0091", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "architecture", "must_mention": ["deepagents", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0092", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "execution_flow", "must_mention": ["deepagents", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0093", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "dependencies", "must_mention": ["deepagents", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0094", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "configuration", "must_mention": ["deepagents", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0095", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "testing", "must_mention": ["deepagents", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0096", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "training_or_core_algorithm", "must_mention": ["deepagents", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0097", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "extension_points", "must_mention": ["deepagents", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0098", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "debugging", "must_mention": ["deepagents", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0099", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "api_comparison", "must_mention": ["deepagents", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0100", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "langchain-ai/deepagents リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "langchain-ai/deepagents", "domain": "長期タスク型エージェント", "task_type": "onboarding", "must_mention": ["deepagents", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0101", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "architecture", "must_mention": ["vllm", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0102", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "execution_flow", "must_mention": ["vllm", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0103", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "dependencies", "must_mention": ["vllm", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0104", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "configuration", "must_mention": ["vllm", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0105", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "testing", "must_mention": ["vllm", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0106", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "training_or_core_algorithm", "must_mention": ["vllm", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0107", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "extension_points", "must_mention": ["vllm", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0108", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "debugging", "must_mention": ["vllm", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0109", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "api_comparison", "must_mention": ["vllm", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0110", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "vllm-project/vllm リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "vllm-project/vllm", "domain": "高速LLMサービング", "task_type": "onboarding", "must_mention": ["vllm", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0111", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "architecture", "must_mention": ["sglang", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0112", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "execution_flow", "must_mention": ["sglang", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0113", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "dependencies", "must_mention": ["sglang", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0114", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "configuration", "must_mention": ["sglang", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0115", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "testing", "must_mention": ["sglang", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0116", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "training_or_core_algorithm", "must_mention": ["sglang", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0117", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "extension_points", "must_mention": ["sglang", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0118", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "debugging", "must_mention": ["sglang", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0119", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "api_comparison", "must_mention": ["sglang", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0120", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "sgl-project/sglang リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "sgl-project/sglang", "domain": "LLMサービング・構造化生成", "task_type": "onboarding", "must_mention": ["sglang", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0121", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "architecture", "must_mention": ["Megatron-LM", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0122", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "execution_flow", "must_mention": ["Megatron-LM", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0123", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "dependencies", "must_mention": ["Megatron-LM", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0124", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "configuration", "must_mention": ["Megatron-LM", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0125", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "testing", "must_mention": ["Megatron-LM", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0126", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "training_or_core_algorithm", "must_mention": ["Megatron-LM", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0127", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "extension_points", "must_mention": ["Megatron-LM", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0128", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "debugging", "must_mention": ["Megatron-LM", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0129", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "api_comparison", "must_mention": ["Megatron-LM", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0130", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "NVIDIA/Megatron-LM リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "NVIDIA/Megatron-LM", "domain": "大規模分散学習", "task_type": "onboarding", "must_mention": ["Megatron-LM", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0131", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "architecture", "must_mention": ["DeepSpeed", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0132", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "execution_flow", "must_mention": ["DeepSpeed", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0133", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "dependencies", "must_mention": ["DeepSpeed", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0134", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "configuration", "must_mention": ["DeepSpeed", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0135", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "testing", "must_mention": ["DeepSpeed", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0136", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "training_or_core_algorithm", "must_mention": ["DeepSpeed", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0137", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "extension_points", "must_mention": ["DeepSpeed", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0138", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "debugging", "must_mention": ["DeepSpeed", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0139", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "api_comparison", "must_mention": ["DeepSpeed", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0140", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "microsoft/DeepSpeed リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "microsoft/DeepSpeed", "domain": "大規模学習最適化", "task_type": "onboarding", "must_mention": ["DeepSpeed", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0141", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "architecture", "must_mention": ["ray", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0142", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "execution_flow", "must_mention": ["ray", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0143", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "dependencies", "must_mention": ["ray", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0144", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "configuration", "must_mention": ["ray", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0145", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "testing", "must_mention": ["ray", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0146", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "training_or_core_algorithm", "must_mention": ["ray", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0147", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "extension_points", "must_mention": ["ray", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0148", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "debugging", "must_mention": ["ray", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0149", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "api_comparison", "must_mention": ["ray", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0150", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ray-project/ray リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ray-project/ray", "domain": "分散コンピューティング", "task_type": "onboarding", "must_mention": ["ray", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0151", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "architecture", "must_mention": ["pytorch", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0152", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "execution_flow", "must_mention": ["pytorch", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0153", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "dependencies", "must_mention": ["pytorch", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0154", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "configuration", "must_mention": ["pytorch", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0155", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "testing", "must_mention": ["pytorch", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0156", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "training_or_core_algorithm", "must_mention": ["pytorch", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0157", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "extension_points", "must_mention": ["pytorch", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0158", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "debugging", "must_mention": ["pytorch", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0159", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "api_comparison", "must_mention": ["pytorch", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0160", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytorch/pytorch リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "pytorch/pytorch", "domain": "深層学習フレームワーク", "task_type": "onboarding", "must_mention": ["pytorch", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0161", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "architecture", "must_mention": ["jax", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0162", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "execution_flow", "must_mention": ["jax", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0163", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "dependencies", "must_mention": ["jax", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0164", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "configuration", "must_mention": ["jax", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0165", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "testing", "must_mention": ["jax", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0166", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "training_or_core_algorithm", "must_mention": ["jax", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0167", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "extension_points", "must_mention": ["jax", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0168", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "debugging", "must_mention": ["jax", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0169", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "api_comparison", "must_mention": ["jax", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0170", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "jax-ml/jax リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "jax-ml/jax", "domain": "高性能数値計算", "task_type": "onboarding", "must_mention": ["jax", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0171", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "architecture", "must_mention": ["openai-python", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0172", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "execution_flow", "must_mention": ["openai-python", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0173", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "dependencies", "must_mention": ["openai-python", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0174", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "configuration", "must_mention": ["openai-python", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0175", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "testing", "must_mention": ["openai-python", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0176", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "training_or_core_algorithm", "must_mention": ["openai-python", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0177", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "extension_points", "must_mention": ["openai-python", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0178", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "debugging", "must_mention": ["openai-python", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0179", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "api_comparison", "must_mention": ["openai-python", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0180", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "openai/openai-python リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "openai/openai-python", "domain": "OpenAI Python SDK", "task_type": "onboarding", "must_mention": ["openai-python", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0181", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "architecture", "must_mention": ["ollama", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0182", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "execution_flow", "must_mention": ["ollama", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0183", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "dependencies", "must_mention": ["ollama", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0184", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "configuration", "must_mention": ["ollama", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0185", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "testing", "must_mention": ["ollama", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0186", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "training_or_core_algorithm", "must_mention": ["ollama", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0187", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "extension_points", "must_mention": ["ollama", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0188", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "debugging", "must_mention": ["ollama", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0189", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "api_comparison", "must_mention": ["ollama", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0190", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ollama/ollama リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ollama/ollama", "domain": "ローカルモデル実行", "task_type": "onboarding", "must_mention": ["ollama", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0191", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "architecture", "must_mention": ["llama.cpp", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0192", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "execution_flow", "must_mention": ["llama.cpp", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0193", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "dependencies", "must_mention": ["llama.cpp", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0194", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "configuration", "must_mention": ["llama.cpp", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0195", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "testing", "must_mention": ["llama.cpp", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0196", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "training_or_core_algorithm", "must_mention": ["llama.cpp", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0197", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "extension_points", "must_mention": ["llama.cpp", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0198", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "debugging", "must_mention": ["llama.cpp", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0199", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "api_comparison", "must_mention": ["llama.cpp", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0200", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "ggerganov/llama.cpp リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "ggerganov/llama.cpp", "domain": "量子化LLM推論", "task_type": "onboarding", "must_mention": ["llama.cpp", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0201", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "architecture", "must_mention": ["litgpt", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0202", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "execution_flow", "must_mention": ["litgpt", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0203", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "dependencies", "must_mention": ["litgpt", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0204", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "configuration", "must_mention": ["litgpt", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0205", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "testing", "must_mention": ["litgpt", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0206", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "training_or_core_algorithm", "must_mention": ["litgpt", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0207", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "extension_points", "must_mention": ["litgpt", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0208", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "debugging", "must_mention": ["litgpt", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0209", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "api_comparison", "must_mention": ["litgpt", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0210", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "Lightning-AI/litgpt リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "Lightning-AI/litgpt", "domain": "LLM学習・推論", "task_type": "onboarding", "must_mention": ["litgpt", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0211", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "architecture", "must_mention": ["axolotl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0212", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "execution_flow", "must_mention": ["axolotl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0213", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "dependencies", "must_mention": ["axolotl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0214", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "configuration", "must_mention": ["axolotl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0215", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "testing", "must_mention": ["axolotl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0216", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "training_or_core_algorithm", "must_mention": ["axolotl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0217", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "extension_points", "must_mention": ["axolotl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0218", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "debugging", "must_mention": ["axolotl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0219", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "api_comparison", "must_mention": ["axolotl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0220", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "axolotl-ai-cloud/axolotl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "axolotl-ai-cloud/axolotl", "domain": "LLMファインチューニング", "task_type": "onboarding", "must_mention": ["axolotl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0221", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "architecture", "must_mention": ["OpenRLHF", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0222", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "execution_flow", "must_mention": ["OpenRLHF", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0223", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "dependencies", "must_mention": ["OpenRLHF", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0224", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "configuration", "must_mention": ["OpenRLHF", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0225", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "testing", "must_mention": ["OpenRLHF", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0226", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "training_or_core_algorithm", "must_mention": ["OpenRLHF", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0227", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "extension_points", "must_mention": ["OpenRLHF", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0228", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "debugging", "must_mention": ["OpenRLHF", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0229", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "api_comparison", "must_mention": ["OpenRLHF", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0230", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "OpenRLHF/OpenRLHF リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "OpenRLHF/OpenRLHF", "domain": "RLHF学習", "task_type": "onboarding", "must_mention": ["OpenRLHF", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0231", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "architecture", "must_mention": ["verl", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0232", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "execution_flow", "must_mention": ["verl", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0233", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "dependencies", "must_mention": ["verl", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0234", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "configuration", "must_mention": ["verl", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0235", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "testing", "must_mention": ["verl", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0236", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "training_or_core_algorithm", "must_mention": ["verl", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0237", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "extension_points", "must_mention": ["verl", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0238", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "debugging", "must_mention": ["verl", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0239", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "api_comparison", "must_mention": ["verl", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0240", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "volcengine/verl リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "volcengine/verl", "domain": "強化学習基盤", "task_type": "onboarding", "must_mention": ["verl", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "train"}
{"id": "deepwiki-0241", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "architecture", "must_mention": ["open-instruct", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0242", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "execution_flow", "must_mention": ["open-instruct", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0243", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "dependencies", "must_mention": ["open-instruct", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0244", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "configuration", "must_mention": ["open-instruct", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0245", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "testing", "must_mention": ["open-instruct", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0246", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "training_or_core_algorithm", "must_mention": ["open-instruct", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0247", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "extension_points", "must_mention": ["open-instruct", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0248", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "debugging", "must_mention": ["open-instruct", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0249", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "api_comparison", "must_mention": ["open-instruct", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0250", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "allenai/open-instruct リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "allenai/open-instruct", "domain": "命令学習", "task_type": "onboarding", "must_mention": ["open-instruct", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0251", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "architecture", "must_mention": ["mlflow", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0252", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "execution_flow", "must_mention": ["mlflow", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0253", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "dependencies", "must_mention": ["mlflow", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0254", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "configuration", "must_mention": ["mlflow", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0255", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "testing", "must_mention": ["mlflow", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0256", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "training_or_core_algorithm", "must_mention": ["mlflow", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0257", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "extension_points", "must_mention": ["mlflow", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0258", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "debugging", "must_mention": ["mlflow", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0259", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "api_comparison", "must_mention": ["mlflow", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0260", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "mlflow/mlflow リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "mlflow/mlflow", "domain": "MLライフサイクル管理", "task_type": "onboarding", "must_mention": ["mlflow", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0261", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "architecture", "must_mention": ["wandb", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0262", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "execution_flow", "must_mention": ["wandb", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0263", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "dependencies", "must_mention": ["wandb", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0264", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "configuration", "must_mention": ["wandb", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0265", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "testing", "must_mention": ["wandb", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0266", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "training_or_core_algorithm", "must_mention": ["wandb", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0267", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "extension_points", "must_mention": ["wandb", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0268", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "debugging", "must_mention": ["wandb", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0269", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "api_comparison", "must_mention": ["wandb", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0270", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "wandb/wandb リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "wandb/wandb", "domain": "実験追跡", "task_type": "onboarding", "must_mention": ["wandb", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "validation"}
{"id": "deepwiki-0271", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "architecture", "must_mention": ["airflow", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0272", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "execution_flow", "must_mention": ["airflow", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0273", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "dependencies", "must_mention": ["airflow", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0274", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "configuration", "must_mention": ["airflow", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0275", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "testing", "must_mention": ["airflow", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0276", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "training_or_core_algorithm", "must_mention": ["airflow", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0277", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "extension_points", "must_mention": ["airflow", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0278", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "debugging", "must_mention": ["airflow", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0279", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "api_comparison", "must_mention": ["airflow", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0280", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "apache/airflow リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "apache/airflow", "domain": "ワークフロー管理", "task_type": "onboarding", "must_mention": ["airflow", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0281", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "architecture", "must_mention": ["fastapi", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0282", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "execution_flow", "must_mention": ["fastapi", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0283", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "dependencies", "must_mention": ["fastapi", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0284", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "configuration", "must_mention": ["fastapi", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0285", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "testing", "must_mention": ["fastapi", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0286", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "training_or_core_algorithm", "must_mention": ["fastapi", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0287", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "extension_points", "must_mention": ["fastapi", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0288", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "debugging", "must_mention": ["fastapi", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0289", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "api_comparison", "must_mention": ["fastapi", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0290", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "fastapi/fastapi リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "fastapi/fastapi", "domain": "Python Web API", "task_type": "onboarding", "must_mention": ["fastapi", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0291", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。主要ディレクトリと中核モジュールの責務を整理し、全体アーキテクチャを説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "architecture", "must_mention": ["pytest", "README", "ディレクトリ", "モジュール"], "evidence_requirements": ["READMEまたはドキュメント", "2つ以上の主要ディレクトリ", "中核モジュールの実ファイル"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0292", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。代表的な利用例を1つ選び、ユーザー入力から主要処理を経て出力に至る実行経路を追跡してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "execution_flow", "must_mention": ["pytest", "エントリーポイント", "実行経路", "出力"], "evidence_requirements": ["公式の利用例", "エントリーポイント", "呼び出し先の実装"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0293", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。依存関係の定義方法を調べ、必須依存・任意依存・開発依存の役割と主要な外部ライブラリを説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "dependencies", "must_mention": ["pytest", "依存関係", "必須", "任意"], "evidence_requirements": ["依存関係を定義する設定ファイル", "必須依存の具体例", "任意または開発依存の具体例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0294", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。設定を読み込んで実行時の挙動へ反映する仕組みを調べ、主要な設定項目と上書き方法を説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "configuration", "must_mention": ["pytest", "設定", "デフォルト", "上書き"], "evidence_requirements": ["設定ファイルまたは設定クラス", "デフォルト値", "CLI・環境変数・コードのいずれかによる上書き経路"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0295", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。テスト構成とCIを調べ、主要機能がどの粒度で検証され、変更時にどのチェックが走るか説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "testing", "must_mention": ["pytest", "テスト", "CI", "検証"], "evidence_requirements": ["テストディレクトリ", "具体的なテストファイル", "CIワークフローまたはテスト設定"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0296", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。このリポジトリの中核アルゴリズムまたは学習・最適化処理を特定し、入力、主要ステップ、出力を実装に沿って説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "training_or_core_algorithm", "must_mention": ["pytest", "入力", "主要ステップ", "出力"], "evidence_requirements": ["中核処理の実装ファイル", "公開APIまたは呼び出し元", "入出力を示すコード"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0297", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。利用者が機能を追加・差し替えできる拡張ポイントを調べ、インターフェースと具体的な拡張例を説明してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "extension_points", "must_mention": ["pytest", "拡張", "インターフェース", "実装例"], "evidence_requirements": ["基底クラス・プロトコル・フックのいずれか", "登録または注入の仕組み", "具体的な実装例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0298", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。典型的な障害を1つ想定し、ログ・例外・検証処理を手掛かりに、原因を切り分ける調査手順を作ってください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "debugging", "must_mention": ["pytest", "障害", "原因", "切り分け"], "evidence_requirements": ["例外または検証処理の実装", "ログ出力箇所", "関連する設定またはドキュメント"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0299", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。目的が近い2つの公開APIまたは実行モードを選び、用途、入力、挙動、使い分けを比較してください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "api_comparison", "must_mention": ["pytest", "比較", "用途", "使い分け"], "evidence_requirements": ["比較対象となる2つの公開APIまたはモード", "各実装または公式ドキュメント", "差が分かる利用例"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}
{"id": "deepwiki-0300", "prompt": [{"role": "system", "content": "あなたはリポジトリ調査エージェントです。DeepWikiを使って対象リポジトリを調査し、根拠となるファイルパス、シンボル名、設定名を明示してください。同じ検索を繰り返さず、推測と確認済み事実を区別し、最後に簡潔な結論を提示してください。"}, {"role": "user", "content": "pytest-dev/pytest リポジトリをDeepWikiで調査してください。新規コントリビューター向けに、最小セットアップ、最初に読むべきコード、実行・テスト方法を含む導入ガイドを作ってください。"}], "repository": "pytest-dev/pytest", "domain": "Pythonテスト", "task_type": "onboarding", "must_mention": ["pytest", "セットアップ", "実行", "テスト"], "evidence_requirements": ["インストール手順", "最小実行例", "テストまたはコントリビューション手順"], "reward_spec": {"keyword_coverage_weight": 2.0, "evidence_coverage_weight": 2.0, "tool_success_weight": 0.3, "duplicate_tool_call_penalty": -0.5, "tool_error_penalty": -1.0, "unsupported_claim_penalty": -1.0}, "max_tool_steps": 8, "language": "ja", "split": "test"}


### 3. 学習コード

In [ ]:
# ============================================================
# STEP 1
# Native TRL + environment_factory + QLoRA
#
# 目的:
#   - Native TRL の GRPOTrainer を使用する
#   - environment_factory により DeepWiki MCP を tool として利用する
#   - bitsandbytes 4-bit + PEFT LoRA により QLoRA 学習を行う
#
# 注意:
#   - このコードでは Unsloth を使用しない
#   - Transformers >= 5.2.0 が必要
#   - MCP tool 名は環境によって変わる可能性があるため起動時に確認する
# ============================================================

import asyncio
import inspect
import json
import re

import bitsandbytes
import peft
import torch
import transformers
import trl

from datasets import load_dataset
from langchain_mcp_adapters.client import MultiServerMCPClient
from packaging.version import Version
from peft import LoraConfig
from transformers import AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from trl import GRPOConfig, GRPOTrainer


# ============================================================
# 0. ライブラリバージョン / capability check
# ============================================================

# 実際に読み込まれたライブラリのバージョンと Trainer を表示し、環境差分を確認する。
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("trl         :", trl.__version__)
print("peft        :", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("GRPOTrainer :", GRPOTrainer)

# Step1 が前提とする Transformers と Native TRL の組み合わせでない場合は学習前に停止する。
assert Version(transformers.__version__) >= Version("5.2.0"), "environment_factory requires transformers>=5.2.0"
assert "Unsloth" not in str(GRPOTrainer), "Step1 は Native TRL を使うため UnslothGRPOTrainer は使用できません。"

# インストール済み Trainer の引数を調べ、環境生成・量子化・LoRA の受け渡し口を検証する。
trainer_signature = inspect.signature(GRPOTrainer.__init__)

assert "environment_factory" in trainer_signature.parameters, "Installed TRL does not support environment_factory."
assert "quantization_config" in trainer_signature.parameters, "Installed TRL does not expose quantization_config."
assert "peft_config" in trainer_signature.parameters, "Installed TRL does not expose peft_config."

# 使用する GRPOConfig のフィールドも起動時に確認する。
config_fields = GRPOConfig.__dataclass_fields__

assert "max_tool_calling_iterations" in config_fields, "GRPOConfig does not support max_tool_calling_iterations."
assert "num_generations" in config_fields, "GRPOConfig does not support num_generations."
assert "max_completion_length" in config_fields, "GRPOConfig does not support max_completion_length."

print("Native TRL capability check: OK")


# ============================================================
# 1. DeepWiki MCP
# ============================================================

# DeepWiki に streamable HTTP で接続するための MCP サーバー設定。
mcp_config = {
    "deepwiki": {
        "url": "https://mcp.deepwiki.com/mcp",
        "transport": "streamable_http",
    }
}

mcp_client = MultiServerMCPClient(mcp_config)

# 通常の .py スクリプトとして実行するため asyncio.run() を使用する。
# Jupyter Notebook では次の1行を `deepwiki_tools = await mcp_client.get_tools()` に置き換えてもよい。
# deepwiki_tools = asyncio.run(mcp_client.get_tools())
# この Notebook ではトップレベル await で tool 一覧を取得する。
deepwiki_tools = await mcp_client.get_tools()

# サーバーから返った実際の tool 名を表示し、後続の lookup 対象を確認する。
print("MCP tools:")

for tool in deepwiki_tools:
    print(" -", tool.name)


# ============================================================
# 2. DeepWiki tool lookup
# ============================================================

def get_available_tool_names():
    # lookup 失敗時のエラー表示用に、取得済み tool の名前だけを集める。
    names = []

    for tool in deepwiki_tools:
        names.append(tool.name)

    return names


def find_mcp_tool(name):
    # 取得済み MCP tool の名前が完全一致するものを返す。
    for tool in deepwiki_tools:
        if tool.name == name:
            return tool

    # 想定名が見つからない場合、利用可能な名前を示して学習前に停止する。
    available_names = get_available_tool_names()
    raise RuntimeError(f"DeepWiki MCP tool {name!r} not found. available={available_names}")


# 公開 method から呼ぶ3つの DeepWiki tool を実際の MCP 応答から解決する。
DW_STRUCTURE = find_mcp_tool("read_wiki_structure")
DW_CONTENTS = find_mcp_tool("read_wiki_contents")
DW_QUESTION = find_mcp_tool("ask_wiki_question")


# ============================================================
# 3. Dataset
# ============================================================

# JSONL 全体を読み込む。ここでの split="train" は datasets 側の読込先名。
dataset = load_dataset("json", data_files="deepwiki_grpo_dataset_300.jsonl", split="train")

# 本学習時はこちらを使用する。
# 実際の学習対象は各レコードの split 列が train の行だけに絞る。
train_dataset = dataset.filter(lambda row: row["split"] == "train")

# スモークテストだけを行いたい場合は上の行をコメントアウトし、
# 以下の2行を有効にする。
# smoke_count = min(20, len(dataset))
# train_dataset = dataset.select(range(smoke_count))

# validation/test は件数を確認するが、このセルの Trainer には渡していない。
validation_dataset = dataset.filter(lambda row: row["split"] == "validation")
test_dataset = dataset.filter(lambda row: row["split"] == "test")

print("train      :", len(train_dataset))
print("validation :", len(validation_dataset))
print("test       :", len(test_dataset))

# environment の初期化と completion reward に必要な列の存在を検査する。
required_columns = {
    "id",
    "prompt",
    "repository",
    "must_mention",
    "evidence_requirements",
    "reward_spec",
    "max_tool_steps",
}

available_columns = set(train_dataset.column_names)
assert required_columns.issubset(available_columns), f"Dataset columns are missing. required={required_columns}, actual={available_columns}"


# ============================================================
# 4. Model / QLoRA config
# ============================================================

MODEL_NAME = "Qwen/Qwen3-4B"

# Qwen3 の tokenizer を読み込み、PAD がない場合だけ EOS を PAD として使う。
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# base model の 4-bit NF4 量子化と bfloat16 計算・double quantization を指定する。
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

# attention と MLP の列挙した射影層に LoRA adapter を適用する設定。
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ============================================================
# 5. DeepWiki Environment
# ============================================================

class DeepWikiEnvironment:
    """
    1 rollout ごとに1インスタンス生成される stateful environment。

    reset() で Dataset の metadata を受け取り、
    public method が TRL により tool として公開される。
    """

    def __init__(self):
        # 各 rollout の tool 履歴、重複検出状態、dataset 由来の設定を初期化する。
        self.records = []
        self.seen_calls = set()
        self.repository = None
        self.reward_spec = {}
        self.max_tool_steps = 8

    # --------------------------------------------------------
    # Reserved method
    # --------------------------------------------------------

    def reset(self, repository=None, reward_spec=None, max_tool_steps=8, **kwargs):
        """Rollout 開始時に environment の内部状態を初期化する。"""

        # dataset metadata を受け取り、前回 rollout の履歴と重複判定を消去する。
        self.records = []
        self.seen_calls = set()
        self.repository = repository
        self.reward_spec = reward_spec or {}
        self.max_tool_steps = int(max_tool_steps or 8)

        # Dataset 側の prompt をそのまま使うため、追加 observation は返さない。
        return None

    def get_reward(self) -> float:
        """Tool trajectory に対する step reward の合計を返す。"""

        # tool を一度も使わない rollout には固定の負 reward を返す。
        if not self.records:
            return -1.0

        # 記録済み call ごとの reward を合算する（成功数への上限は設けていない）。
        total_reward = 0.0

        for record in self.records:
            total_reward += float(record["reward"])

        return float(total_reward)

    # --------------------------------------------------------
    # Private helper
    # --------------------------------------------------------

    async def _execute(self, tool, tool_name, args):
        """MCP tool を実行し、成功・重複・エラーに応じて step reward を記録する。"""

        # 現在の記録件数を call の step 番号として使う。
        step = len(self.records)

        # Dataset で指定された tool step 上限を超えた場合。
        if step >= self.max_tool_steps:
            observation = f"ERROR: maximum tool steps {self.max_tool_steps} exceeded"
            reward = float(self.reward_spec.get("tool_error_penalty", -1.0))

            # MCP は呼ばず、上限エラーを失敗 record として保存する。
            record = {
                "step": step,
                "tool": tool_name,
                "args": args,
                "observation": observation,
                "success": False,
                "duplicate": False,
                "reward": reward,
            }

            self.records.append(record)
            return observation

        # 同一 tool + args の完全重複を検出する。
        call_data = {
            "tool": tool_name,
            "args": args,
        }

        # キー順を揃えた JSON 文字列で比較し、呼び出し済みのキーを追加する。
        call_key = json.dumps(call_data, ensure_ascii=False, sort_keys=True)
        duplicate = call_key in self.seen_calls
        self.seen_calls.add(call_key)

        try:
            # tool を非同期実行し、文字列以外の結果も observation 用に文字列化する。
            result = await tool.ainvoke(args)

            if isinstance(result, str):
                observation = result
            else:
                observation = json.dumps(result, ensure_ascii=False, default=str)

            success = True

        except Exception as exc:
            # 例外は observation に変換し、失敗として reward 計算に渡す。
            observation = f"ERROR: {type(exc).__name__}: {exc}"
            success = False

        # 成功/失敗の step reward に、重複していれば追加の penalty を加算する。
        reward = 0.0

        if success:
            reward += float(self.reward_spec.get("tool_success_weight", 0.3))
        else:
            reward += float(self.reward_spec.get("tool_error_penalty", -1.0))

        if duplicate:
            reward += float(self.reward_spec.get("duplicate_tool_call_penalty", -0.5))

        # call の引数、観測結果、成否、重複、reward を後続の集計用に記録する。
        record = {
            "step": step,
            "tool": tool_name,
            "args": args,
            "observation": observation,
            "success": success,
            "duplicate": duplicate,
            "reward": reward,
        }

        self.records.append(record)
        return observation

    def _get_observations(self):
        """Reward function から参照するため、成功した tool observation を連結する。"""

        observations = []

        # 失敗時の observation は evidence/unsupported-claim 判定には含めない。
        for record in self.records:
            if record["success"]:
                observations.append(record["observation"])

        return "\n\n".join(observations)

    # --------------------------------------------------------
    # PUBLIC METHODS
    # 以下の method は TRL により tool として公開される。
    # --------------------------------------------------------

    async def read_wiki_structure(self, repoName: str) -> str:
        """
        Read the DeepWiki documentation structure of a GitHub repository.

        Args:
            repoName: Repository name in owner/repository form.

        Returns:
            The repository documentation structure returned by DeepWiki.
        """

        # public method の英語 docstring は TRL の tool schema に利用する。
        args = {"repoName": repoName}
        return await self._execute(DW_STRUCTURE, "read_wiki_structure", args)

    async def read_wiki_contents(self, repoName: str) -> str:
        """
        Read the DeepWiki documentation contents of a GitHub repository.

        Args:
            repoName: Repository name in owner/repository form.

        Returns:
            Repository documentation returned by DeepWiki.
        """

        # MCP tool に渡す repoName をそのまま args に格納して共通処理に委譲する。
        args = {"repoName": repoName}
        return await self._execute(DW_CONTENTS, "read_wiki_contents", args)

    async def ask_wiki_question(self, repoName: str, question: str) -> str:
        """
        Ask DeepWiki a specific question about a GitHub repository.

        Args:
            repoName: Repository name in owner/repository form.
            question: Specific repository question to investigate.

        Returns:
            DeepWiki's answer to the question.
        """

        # リポジトリ名と調査質問を指定して共通の MCP 実行処理を呼ぶ。
        args = {
            "repoName": repoName,
            "question": question,
        }

        return await self._execute(DW_QUESTION, "ask_wiki_question", args)


def environment_factory():
    """TRL が rollout ごとに新しい environment instance を作る factory。"""

    # Trainer にはインスタンスではなく、この生成関数を渡す。
    return DeepWikiEnvironment()


# ============================================================
# 6. Environment sanity check
# ============================================================

# 実際に環境を生成して method を列挙し、意図しない helper の tool 公開を検出する。
env_probe = DeepWikiEnvironment()
public_methods = []

for name, member in inspect.getmembers(env_probe, predicate=inspect.ismethod):
    is_reserved = name == "reset" or name == "get_reward"
    is_private = name.startswith("_")

    if not is_reserved and not is_private:
        public_methods.append(name)

print("Methods exposed as tools:", public_methods)

# reserved method と private helper を除く公開 method は3つだけのはず。
expected_public_methods = {
    "read_wiki_structure",
    "read_wiki_contents",
    "ask_wiki_question",
}

assert set(public_methods) == expected_public_methods, f"Unexpected public methods. actual={public_methods}"


# ============================================================
# 7. Reward helpers
# ============================================================

def completion_to_text(completion):
    """Conversational completion から最後の assistant text を抽出する。"""

    # すでにテキストであれば変換せず使う。
    if isinstance(completion, str):
        return completion

    # メッセージ列以外は文字列化して返す。
    if not isinstance(completion, list):
        return str(completion)

    # conversational 形式なら末尾から探し、最後の assistant の文字列 content を返す。
    for msg in reversed(completion):
        if isinstance(msg, dict) and msg.get("role") == "assistant":
            content = msg.get("content", "")

            if isinstance(content, str):
                return content

    return ""


# 最終回答内の拡張子付きファイルパスを抽出する簡易的な正規表現。
FILE_RE = re.compile(r"\b[\w.\-]+(?:/[\w.\-]+)+\.(?:py|toml|yaml|yml|json|md|sh|js|ts|rs|go)\b", re.I)


def evidence_satisfied(requirement, answer, observations):
    """自然言語の evidence requirement を簡易ルールで評価する。"""

    # final answer と成功した tool observation の両方を小文字化して照合する。
    req = requirement.lower()
    combined = (answer + "\n" + observations).lower()

    # README の明示的な言及を確認する。
    if "readme" in req:
        return "readme" in combined

    # 2ディレクトリ以上という要求では、重複を除いたディレクトリ表記を数える。
    if "2つ以上" in requirement and "ディレクトリ" in requirement:
        directory_matches = re.findall(r"\b[\w.\-]+/", combined)
        directories = set(directory_matches)
        return len(directories) >= 2

    implementation_keywords = ["実ファイル", "設定ファイル", "エントリーポイント", "実装"]

    # 実装・設定等の要求は、結合テキスト中にファイルパスがあるかで近似する。
    for keyword in implementation_keywords:
        if keyword in requirement:
            return bool(FILE_RE.search(combined))

    if "利用例" in requirement:
        # 利用例に関係する語のいずれかがあれば満たしたと判定する。
        usage_keywords = ["example", "examples", "usage", "quickstart", "利用例", "使用例"]

        for keyword in usage_keywords:
            if keyword in combined:
                return True

        return False

    if "必須依存" in requirement:
        # 依存関係の記述や設定ファイル名を手掛かりにする。
        dependency_keywords = ["dependencies", "requirements", "pyproject.toml", "setup.py"]

        for keyword in dependency_keywords:
            if keyword in combined:
                return True

        return False

    if "任意" in requirement or "開発依存" in requirement:
        # optional / dev dependency もキーワードベースで近似する。
        optional_keywords = ["optional", "dev", "extras", "dependency-groups"]

        for keyword in optional_keywords:
            if keyword in combined:
                return True

        return False

    # その他の要求は句読点・空白で分割し、2文字以上の語の一致を探す。
    raw_tokens = re.split(r"[、。\s]+", requirement)
    tokens = []

    for token in raw_tokens:
        if len(token) >= 2:
            tokens.append(token)

    for token in tokens:
        if token.lower() in combined:
            return True

    return False


# ============================================================
# 8. Keyword reward
# ============================================================

def keyword_coverage_reward(completions, must_mention, reward_spec, **kwargs):
    """must_mention の語句が最終回答に含まれている割合を reward にする。"""

    output = []

    # 同じ index の completion、必須語句、重み設定をまとめて評価する。
    for completion, keywords, spec in zip(completions, must_mention, reward_spec, strict=True):
        text = completion_to_text(completion).lower()
        keywords = keywords or []

        # 必須語句が空なら coverage=1、それ以外は出現数 / 必須語句数。
        if not keywords:
            coverage = 1.0
        else:
            hits = 0

            for keyword in keywords:
                if str(keyword).lower() in text:
                    hits += 1

            coverage = hits / len(keywords)

        # dataset の重みを掛け、completion ごとの reward を返す。
        weight = float(spec.get("keyword_coverage_weight", 2.0))
        reward = coverage * weight
        output.append(reward)

    return output


# ============================================================
# 9. Evidence reward
# ============================================================

def evidence_coverage_reward(completions, evidence_requirements, reward_spec, environments=None, **kwargs):
    """Dataset の evidence_requirements をどれだけ満たすかを reward にする。"""

    output = []

    # 同じ rollout の要件と environment に記録された成功 observation を参照する。
    for index, values in enumerate(zip(completions, evidence_requirements, reward_spec, strict=True)):
        completion = values[0]
        requirements = values[1]
        spec = values[2]
        answer = completion_to_text(completion)
        observations = ""

        if environments is not None:
            observations = environments[index]._get_observations()

        requirements = requirements or []

        # 要件なしは満点、それ以外は簡易ルールで満たした件数の割合を求める。
        if not requirements:
            coverage = 1.0
        else:
            satisfied_count = 0

            for requirement in requirements:
                if evidence_satisfied(requirement, answer, observations):
                    satisfied_count += 1

            coverage = satisfied_count / len(requirements)

        weight = float(spec.get("evidence_coverage_weight", 2.0))
        reward = coverage * weight
        output.append(reward)

    return output


# ============================================================
# 10. Unsupported claim reward
# ============================================================

def unsupported_claim_reward(completions, reward_spec, environments=None, **kwargs):
    """最終回答に記載したファイルパスが tool observation に存在しない場合に penalty を与える。"""

    rewards = []

    for index, values in enumerate(zip(completions, reward_spec, strict=True)):
        completion = values[0]
        spec = values[1]
        answer = completion_to_text(completion)

        # environment が渡されない場合は照合不能なので penalty を付けない。
        if environments is None:
            rewards.append(0.0)
            continue

        # 回答内の重複を除いたファイルパスを成功 observation と照合する。
        observations = environments[index]._get_observations()
        path_matches = FILE_RE.findall(answer)
        paths = set(path_matches)

        if not paths:
            rewards.append(0.0)
            continue

        # observation に文字列として現れない path の割合に penalty を掛ける簡易判定。
        unsupported = 0

        for path in paths:
            if path not in observations:
                unsupported += 1

        ratio = unsupported / len(paths)
        penalty = float(spec.get("unsupported_claim_penalty", -1.0))
        reward = ratio * penalty
        rewards.append(reward)

    return rewards


# ============================================================
# 11. Basic answer quality reward
# ============================================================

def answer_quality_reward(completions, **kwargs):
    """空回答、極端に短い回答を抑制し、結論節を軽く評価する。"""

    rewards = []

    for completion in completions:
        text = completion_to_text(completion).strip()
        reward = 0.0

        # 空回答・100文字未満を減点し、それ以外の回答に小さな加点をする。
        if not text:
            reward -= 1.0
        elif len(text) < 100:
            reward -= 0.3
        else:
            reward += 0.2

        # 結論を示す日本語の見出し語があれば追加で少し加点する。
        has_summary = False
        summary_keywords = ["結論", "まとめ", "要約"]

        for keyword in summary_keywords:
            if keyword in text:
                has_summary = True
                break

        if has_summary:
            reward += 0.1

        rewards.append(reward)

    return rewards


# ============================================================
# 12. Logging callback
# ============================================================

class PrintGRPOCallback(TrainerCallback):
    """GRPO の主要メトリクスだけを Notebook / stdout に表示する。"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        # ログが未提供なら何も表示しない。
        if not logs:
            return

        keys = []

        # loss/reward/reward_std/KL と reward・tool・completion 系だけに絞る。
        for key in logs:
            is_primary = key == "loss" or key == "reward" or key == "reward_std" or key == "kl"
            is_reward = key.startswith("rewards/")
            is_tool = key.startswith("tools/")
            is_completion = key.startswith("completions/")

            if is_primary or is_reward or is_tool or is_completion:
                keys.append(key)

        if not keys:
            return

        # 選択されたメトリクスを Trainer の global step とともに表示する。
        print(f"\n===== GRPO STEP {state.global_step} =====")

        for key in keys:
            print(f"{key:50s} {logs[key]}")


# ============================================================
# 13. GRPOConfig
# ============================================================

# effective batch = 1 * 2 = 2
# num_generations = 2
# group 内の generation 数で割り切れる設定にする。
# 2 epoch、bf16、checkpointing、学習中のログ/保存を GRPOTrainer に渡す。
# max_tool_calling_iterations は Trainer 側の tool 呼び出し上限で、dataset の上限は environment も判定する。
training_args = GRPOConfig(
    output_dir="./outputs/step1-native-trl-qlora",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    warmup_steps=0.05,
    num_generations=2,
    num_generations_eval=2,
    temperature=0.8,
    top_p=0.95,
    max_completion_length=1536,
    max_tool_calling_iterations=8,
    beta=0.02,
    loss_type="dapo",
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    remove_unused_columns=False,
    logging_strategy="steps",
    logging_steps=1,
    logging_first_step=True,
    log_completions=True,
    num_completions_to_print=1,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    seed=42,
)


# ============================================================
# 14. Trainer
# ============================================================

# 4つの completion-level reward を指定する。tool step reward は environment が返す。
reward_functions = [
    keyword_coverage_reward,
    evidence_coverage_reward,
    unsupported_claim_reward,
    answer_quality_reward,
]

# on_log で主要メトリクスを表示する callback を登録する。
callbacks = [PrintGRPOCallback()]

# データ、環境 factory、量子化、LoRA を Native TRL の Trainer にまとめて渡す。
# multi-turn tool calling と GRPO の学習ループは GRPOTrainer が管理する。
trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_functions,
    environment_factory=environment_factory,
    quantization_config=bnb_config,
    peft_config=peft_config,
    callbacks=callbacks,
)


# ============================================================
# 15. QLoRA sanity check
# ============================================================

# Trainer がロードしたモデルが実際に 4-bit か確認する。
is_loaded_in_4bit = getattr(trainer.model, "is_loaded_in_4bit", False)

print("is_loaded_in_4bit:", is_loaded_in_4bit)

assert is_loaded_in_4bit, "Model is not loaded in 4-bit. QLoRA is not active."

# モデル全体と requires_grad=True のパラメータ数を集計する。
trainable = 0
total = 0

for parameter in trainer.model.parameters():
    parameter_count = parameter.numel()
    total += parameter_count

    if parameter.requires_grad:
        trainable += parameter_count

trainable_ratio = 100.0 * trainable / total

print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable_ratio:.4f}%)")

# trainable が全体の 10% 未満でなければ PEFT/QLoRA 設定を疑って停止する。
assert trainable < total * 0.1, "Too many parameters are trainable; PEFT/QLoRA configuration may be wrong."


# ============================================================
# 16. Train
# ============================================================

# ここで学習を起動する。rollout・tool 呼び出し・最適化は Trainer に委譲する。
trainer.train()


# ============================================================
# 17. Save LoRA adapter
# ============================================================

# 最終成果物の保存先。学習中の checkpoint は GRPOConfig の save_steps で管理する。
FINAL_OUTPUT_DIR = "./outputs/step1-native-trl-qlora/final"

# Trainer 経由でモデル（LoRA adapter）を保存し、対応する tokenizer も保存する。
trainer.save_model(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)

print("Saved:", FINAL_OUTPUT_DIR)
